In [2]:
import numpy as np
u = np.array([3.0, 1.0]); v = np.array([1.0, 2.0])

print(u + v)          # [4. 3.]
print(2 * u)          # [6. 2.]
print(v - u)          # [-2.  1.]  the arrow from u's tip to v's tip
print(1.5 * u + 1 * v)  # a linear combination

[4. 3.]
[6. 2.]
[-2.  1.]
[5.5 3.5]


In [3]:
# Mean of many vectors = centroid
pts = np.array([[1, 2], [3, 4], [5, 0]], dtype=float)
print(pts.mean(axis=0))   # [3. 2.]  (average down each column)

[3. 2.]


In [4]:
# Line segment from a to b: convex combinations
a, b = np.array([0.0, 0.0]), np.array([4.0, 2.0])
for t in [0, 0.25, 0.5, 1]:
    print(t, (1 - t) * a + t * b)

0 [0. 0.]
0.25 [1.  0.5]
0.5 [2. 1.]
1 [4. 2.]


In [5]:
import pandas as pd
pl = pd.read_csv("data/pl_matches.csv", low_memory=False)

rows = []
for season, d in pl.groupby("Season"):
    for team in set(d.HomeTeam):
        h, a = d[d.HomeTeam == team], d[d.AwayTeam == team]
        w = (h.FTR == "H").sum() + (a.FTR == "A").sum()
        dr = (h.FTR == "D").sum() + (a.FTR == "D").sum()
        rows.append([w, dr, 3 * w + dr])
V = np.array(rows, dtype=float)

p1, p2 = np.array([1, 0, 3.0]), np.array([0, 1, 1.0])
recon = V[:, [0]] * p1 + V[:, [1]] * p2       # w*p1 + d*p2 for every row (broadcasting)
print(np.allclose(recon, V))                   # True: all 300 points are in the plane
print(V.mean(axis=0))                          # centroid ≈ (14.4, 9.1, 52.4)

True
[14.43666667  9.12666667 52.43666667]


The centroid is the "average team": about 14.4 wins, 9.1 draws and 52.4 points. 

Find the team-season with the highest cosine similarity to Arsenal 2023–24 and the one with the lowest (most opposite style). 

In [6]:
import pandas as pd
pl = pd.read_csv("data/pl_matches.csv", low_memory=False)
rows, names = [], []
for season, d in pl.groupby("Season"):
    for team in sorted(set(d.HomeTeam)):
        h, a = d[d.HomeTeam == team], d[d.AwayTeam == team]
        f = [(h.HS.sum() + a.AS.sum()), (h.HST.sum() + a.AST.sum()), (h.HC.sum() + a.AC.sum()),
             (h.HF.sum() + a.AF.sum()), (h.HY.sum() + a.AY.sum())]
        rows.append(np.array(f) / 38); names.append(f"{team} {season}")
X = np.array(rows); names = np.array(names)
Z = (X - X.mean(axis=0)) / X.std(axis=0)
Zn = Z / np.linalg.norm(Z, axis=1, keepdims=True)     # each row now has length 1
i = list(names).index("Arsenal 2023-24")
sims = Zn @ Zn[i]                                      # cosine similarity with every team-season
order = np.argsort(sims)
print("most similar:", names[order[-2]], round(sims[order[-2]], 3))
print("most opposite:", names[order[0]], round(sims[order[0]], 3))

most similar: Chelsea 2019-20 0.992
most opposite: Stoke 2011-12 -0.976
